# Iterative Refinement

generator/critic LoopAgent with exit_loop.

> Part of the **ADK Katas**. Work top-to-bottom: read the concept, fill in the
> `# TODO`s in the starter cell, then run the **Check** cell — it grades your
> code offline (no API call). The **Run it live** cell needs a `GOOGLE_API_KEY`.


## Kata 17 — Iterative Refinement

A **`LoopAgent`** runs its sub-agents repeatedly until one calls **`exit_loop`**
(or `max_iterations` is hit). The classic pattern is **generator → critic**:
the generator drafts, the critic judges and either asks for changes or exits.

```python
from google.adk.agents import LoopAgent
from google.adk.tools import exit_loop
```

The critic gets `exit_loop` as a tool; calling it sets `escalate` and ends
the loop.

## Your task

Build:
- `generator` — drafts text into `output_key="draft"`.
- `critic` — reviews `state['draft']`; if good enough it calls `exit_loop`,
  else it gives feedback. Give it `tools=[exit_loop]`.
- `root_agent = LoopAgent(name="refine_loop", sub_agents=[generator, critic],
  max_iterations=3)`.

In [ ]:
# Setup — run me first
import sys, pathlib
# Make the kata library importable whether opened from adk-katas/ or adk-katas/notebooks/
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "kata_helpers.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from kata_helpers import *          # load_env, has_api_key, run_agent, check, grade, requires_key
from kata_content import BY_SLUG

KATA = BY_SLUG["iterative-refinement"]
load_env()
print("API key configured:" , has_api_key())


In [ ]:
# ✏️ Your code — fill in the TODOs
from google.adk.agents import Agent, LoopAgent
from google.adk.tools import exit_loop

M = "gemini-2.5-flash"

# TODO: generator (output_key="draft")
generator = None
# TODO: critic (tools=[exit_loop])
critic = None
# TODO: root_agent = LoopAgent(..., max_iterations=3)
root_agent = None

In [ ]:
# ✅ Check your work (offline — grades the symbols you defined above)
results = KATA.run_checks(globals())
grade([check(r.label, r.passed, r.detail) for r in results])


In [ ]:
# ▶️ Run it live (needs GOOGLE_API_KEY). Each run is a fresh single-turn session.
agent = globals().get(KATA.chat_symbol)

if not KATA.chattable:
    print("This kata has no chat agent — the Check cell is the goal. 🎯")
elif has_api_key() and agent is not None:
    result = await run_agent(agent, 'Topic: why backups matter.')   # noqa: F704  (top-level await is fine in Jupyter)
    print("Response:", result.text)
    if result.tool_calls:
        print("Tools called:", result.tool_calls, result.tool_args)
    if result.transfers:
        print("Transferred to:", result.transfers)
    if result.state:
        print("Session state:", result.state)
else:
    requires_key(lambda: None)


---
### Stuck? Reveal the reference solution

<details>
<summary>Show solution</summary>

```python
from google.adk.agents import Agent, LoopAgent
from google.adk.tools import exit_loop

M = "gemini-2.5-flash"

generator = Agent(
    name="generator", model=M,
    instruction="Write or revise a short paragraph on the user's topic. "
                "If feedback is in state, address it.",
    output_key="draft",
)
critic = Agent(
    name="critic", model=M,
    instruction="Review state['draft']. If it is clear and complete, call "
                "exit_loop. Otherwise give one concrete improvement.",
    tools=[exit_loop],
)
root_agent = LoopAgent(
    name="refine_loop",
    sub_agents=[generator, critic],
    max_iterations=3,
)
```

</details>

When you're done, try the same kata in the React app's live chat (`./dev.sh`
from the repo root) to watch the tool-call traces.
